# Downloading Sentinel-2 data (Copernicus Data Space)

*Section 12 — Downloading data, notebook 6 of 8.* Previous: `05_copernicus_odata_sen1`. Next: `07_planet_psscene`.

**You will learn:**

- how to get an access token for the Copernicus Data Space Ecosystem (CDSE) with `rsgispy.dataaccess.copernicus_odata`
- how to query a single Sentinel-2 scene by name, and search for scenes by region, date, cloud cover and product level
- how to download one or many scenes

**Before you start:** a Copernicus Data Space Ecosystem account (https://dataspace.copernicus.eu). The queries do not need an account; the downloads do. Store the username and password in an encoded credentials file (`cop_userinfo.txt`, see *Username and password* below).

**Data:** downloaded from the Copernicus Data Space Ecosystem (OData API) into `outputs/`.

**Run time:** depends on the download; the search cells take about 1 minute.

> **Running this notebook:** every cell that logs in, queries the remote
> service or downloads data is tagged `skip-execution`, so the tutorial test
> runner does not run them. Run them yourself, in order, once you have an
> account and have created the credentials file.

This tutorial uses the Copernicus Data Space Ecosystem OData API to find and
download Sentinel-2 data.

To have access to the Copernicus Data Space Ecosystem services, you need to
register (https://dataspace.copernicus.eu).

## Import modules

In [1]:
import datetime
import os

import rsgispy.tools.credentials
from rsgispy.dataaccess import copernicus_odata

## Create output directory

In [2]:
out_dir_path = os.path.join("outputs", "sen2_data")
os.makedirs(out_dir_path, exist_ok=True)

# The encoded username / password file (see below).
cred_file = "cop_userinfo.txt"

## Username and password

Never write your username and password into a notebook. rsgispy provides a
simple encoding of the username and password so that they are not stored as
plain text. Note that the encoding (base64) is **not** secure: keep the file
private and never commit it to version control.

Create the encoded file with the `rsgisuserpassfile` command-line tool, which
is installed with rsgispy and prompts for the password:

`rsgisuserpassfile -o cop_userinfo.txt -u <your Copernicus Data Space username>`

or, from Python, with
`rsgispy.tools.credentials.create_username_password_file(username, password, out_file)`.
Once you have created `cop_userinfo.txt` (in this folder, or change the path
below), read it into your notebook/script with `get_username_password`.
Alternatively, define the `RSGIS_COP_USER` and `RSGIS_COP_PASS` environment variables
and pass `None` for the username and password.

*RSGISLib users:* the tool was `rsgisuserpassfile.py <file>`; the output file
is now given with `-o`.

In [ ]:
username, password = rsgispy.tools.credentials.get_username_password(
    input_file=cred_file
)

## Access token

When you come to download data, you need an access token to authenticate. The
access token is generated using the following function. Note the token
expires after a short time (about 10 minutes), so create a new one if a
download fails with an authentication error.

In [ ]:
acc_token = copernicus_odata.get_access_token(username=username, password=password)

## Query an individual scene

You can query for an individual scene if you know the name of the scene, such
as below:

In [ ]:
scn_info = copernicus_odata.query_scn(
    scn_name="S2A_MSIL2A_20230904T113321_N0509_R080_T30UVD_20230904T145956.SAFE"
)

In [ ]:
scn_info

In [ ]:
copernicus_odata.download_scn(
    access_token=acc_token, scn_info=scn_info, out_path=out_dir_path
)

## Query multiple scenes

You can query for multiple scenes using the function call below. This allows

 * A bounding box region of interest to be specified
 * Time period start and end date
 * Cloud cover: < 30 %
 * Product type: Level-2A (i.e., surface reflectance)

In [ ]:
scns_info = copernicus_odata.query_scn_lst(
    sensor=copernicus_odata.RSGIS_ODATA_SEN2,
    bbox=(-4.1, -4.0, 52.4, 52.5),
    start_date=datetime.datetime(year=2024, month=1, day=1),
    end_date=datetime.datetime(year=2024, month=6, day=30),
    cloud_cover=30,
    orbit_dir=None,
    product_type=copernicus_odata.RSGIS_ODATA_PROD_TYPE_S2_MSI_2A,
    order_by=copernicus_odata.RSGIS_ODATA_ORDERBY_ASC,
    max_n_rslts=10,
    start_n=None,
)

In [ ]:
scns_info

## Download multiple scenes

In [ ]:
copernicus_odata.download_scns(
    access_token=acc_token,
    scns_info=scns_info,
    out_path=out_dir_path,
    no_except=True,
)